# Figure 2: IQM vendor plots


In [6]:
# --- Setup: libraries and config ---
library(tidyverse)
library(arrow)
library(cowplot)
library(jsonlite)
library(fs)

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) {
  stop("Could not locate config.json. Set CONFIG_PATH or run from the project tree.")
}
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)
source(fs::path(project_root, "scripts", "utils", "plot_style.R"))

# --- Theme and typography ---
plot_style <- get_plot_style(config)
plot_style$axis_line_width <- 0.20
plot_style$axis_tick_width <- 0.20
plot_style$axis_tick_length_pt <- 1.8
font_family_use <- get_export_font_family()
plot_style$font_family <- font_family_use

# Fixed typography for 88 mm assembled figure output.
axis_title_pt <- 5
axis_text_pt <- 5
plot_title_pt <- 6
legend_title_pt <- 5
legend_text_pt <- 5
panel_tag_pt <- 6

theme_pub <- make_theme_pub(
  style = plot_style,
  axis_title_pt = axis_title_pt,
  axis_text_pt = axis_text_pt,
  plot_title_pt = plot_title_pt,
  legend_title_pt = legend_title_pt,
  legend_text_pt = legend_text_pt,
  legend_position = 'none',
  base_size_pt = 5
) +
  theme(
    plot.title = element_text(face = 'plain', margin = margin(0, 0, 0, 0)),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.margin = margin(0, 0, 0, 0)
  )

top_strip_ratio <- 0.22
right_strip_ratio <- 0.16
box_right_strip_ratio <- 0.00
box_top_strip_ratio <- 0.08
vendor_order <- c("GE", "Philips", "Siemens")
figure2_dir <- fs::path(project_root, "figures", "Figure2")
fs::dir_create(fs::path(figure2_dir, "panels"), recurse = TRUE)

# Target final assembled Figure 2 size (for Illustrator assembly).
assembled_width_mm <- get_figure_dims(layout = 'one_column', height_mm = 100)$width_mm
base_figure_dims <- get_figure_dims(layout = 'two_column', height_mm = 160)
assembled_height_mm <- base_figure_dims$height_mm * (assembled_width_mm / base_figure_dims$width_mm)
assembled_figure_dims <- list(width_mm = assembled_width_mm, height_mm = assembled_height_mm)
panel_cell_dims <- list(width_mm = assembled_figure_dims$width_mm / 2, height_mm = assembled_figure_dims$height_mm / 2)
# In the combined layout, panel D occupies half of its right-hand cell (the other half is spacer).
panel_d_dims_final <- list(width_mm = panel_cell_dims$width_mm / 2, height_mm = panel_cell_dims$height_mm)

# --- Data ---
data_candidates <- c(
  file.path(project_root, 'data', 'harmonized_data', 'merged_data_meisler_analyses_harmonized.parquet'),
  file.path(project_root, 'data', 'raw_data', 'merged_data_meisler_analyses.parquet')
)
data_path <- data_candidates[file.exists(data_candidates)][1]
if (is.na(data_path) || !nzchar(data_path)) stop('No parquet data file found in expected locations.')
df <- read_parquet(data_path) %>%
  as_tibble() %>%
  mutate(scanner_manufacturer = factor(as.character(scanner_manufacturer), levels = vendor_order))

required_cols <- c(
  'scanner_manufacturer',
  'raw_neighbor_corr', 't1post_neighbor_corr',
  'raw_dwi_contrast', 't1post_dwi_contrast',
  'CNR0_median', 'CNR1_median', 'CNR2_median', 'CNR3_median', 'CNR4_median',
  'batch_device_software', 'age'
)
missing_cols <- setdiff(required_cols, names(df))
if (length(missing_cols) > 0) stop(paste('Missing required columns:', paste(missing_cols, collapse = ', ')))

# --- Export helper (mm-based PDF for this figure) ---
save_pdf_arial <- function(path, plot, width_mm, height_mm) {
  ggsave(
    filename = path,
    plot = plot,
    width = width_mm,
    height = height_mm,
    units = 'mm',
    device = cairo_pdf,
    family = font_family_use
  )
}

# --- Panel helper functions ---
add_panel_tag <- function(panel_plot, tag_text) {
  tag_layer <- ggplot() +
    annotate(
      'text',
      x = 0.01,
      y = 0.995,
      label = tag_text,
      hjust = 0,
      vjust = 1,
      family = plot_style$font_family,
      fontface = 'bold',
      size = panel_tag_pt / .pt
    ) +
    coord_cartesian(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE, clip = 'off') +
    theme_void() +
    theme(plot.margin = margin(0, 0, 0, 0))

  cowplot::ggdraw() +
    cowplot::draw_plot(panel_plot, 0, 0, 1, 1) +
    cowplot::draw_plot(tag_layer, 0, 0, 1, 1)
}

build_scatter_with_marginals <- function(data, xvar, yvar, xlab, ylab, fixed_upper = NULL) {
  dd <- data %>%
    select(scanner_manufacturer, x = all_of(xvar), y = all_of(yvar)) %>%
    filter(!is.na(scanner_manufacturer), !is.na(x), !is.na(y))

  if (nrow(dd) == 0) stop(paste('No data for', xvar, 'vs', yvar))

  xlim <- range(dd$x, na.rm = TRUE)
  ylim <- range(dd$y, na.rm = TRUE)
  if (!is.null(fixed_upper)) {
    xlim[2] <- fixed_upper
    ylim[2] <- fixed_upper
  }

  # Single strip size used in both directions so top/right marginals match.
  strip_ratio <- 0.12
  p_scatter <- ggplot(dd, aes(x = x, y = y, color = scanner_manufacturer)) +
    geom_abline(intercept = 0, slope = 1, linetype = 'dotted', linewidth = 0.28, color = 'black') +
    geom_point(alpha = 0.35, size = 0.55, stroke = 0) +
    scale_color_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    coord_cartesian(xlim = xlim, ylim = ylim, expand = FALSE) +
    labs(x = xlab, y = ylab) +
    theme_pub +
    theme(
      axis.line = element_blank(),
      panel.border = element_rect(color = 'black', fill = NA, linewidth = 0.20),
      aspect.ratio = 1,
      legend.position = 'none',
      plot.margin = margin(0, 0, 0, 0)
    )

  p_top <- ggplot(dd, aes(x = x, y = after_stat(ifelse(scaled < 0.015, NA_real_, scaled)), color = scanner_manufacturer, fill = scanner_manufacturer)) +
    geom_density(alpha = 0.20, linewidth = 0.28) +
    scale_color_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_y_continuous(limits = c(0, 1.02), breaks = c(0.75, 0.85, 0.95), labels = c('0.75', '0.85', '0.95'), expand = expansion(mult = c(0, 0))) +
    coord_cartesian(xlim = xlim, expand = FALSE) +
    labs(y = NULL) +
    theme_pub +
    theme(
      axis.title.x = element_blank(),
      axis.text.x = element_blank(),
      axis.ticks.x = element_blank(),
      axis.line.x = element_blank(),
      axis.title.y = element_text(color = NA),
      axis.text.y = element_text(color = NA),
      axis.ticks.y = element_line(color = NA),
      axis.line.y = element_blank(),
      panel.border = element_blank(),
      legend.position = 'none',
      plot.margin = margin(0, 0, -2, 0)
    )

  p_right <- ggplot(dd, aes(x = y, y = after_stat(ifelse(scaled < 0.015, NA_real_, scaled)), color = scanner_manufacturer, fill = scanner_manufacturer)) +
    geom_density(alpha = 0.20, linewidth = 0.28) +
    scale_color_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
    scale_y_continuous(limits = c(0, 1.02), expand = expansion(mult = c(0, 0))) +
    coord_flip(xlim = ylim, expand = FALSE) +
    theme_void(base_family = plot_style$font_family) +
    theme(
      legend.position = 'none',
      plot.margin = margin(0, 0, 0, -2)
    )

  p_blank <- ggplot() +
    theme_void() +
    theme(plot.margin = margin(0, 0, 0, 0))

  top_row <- cowplot::plot_grid(
    p_top,
    p_blank,
    nrow = 1,
    align = 'h',
    axis = 'tb',
    rel_widths = c(1, strip_ratio)
  )

  bottom_row <- cowplot::plot_grid(
    p_scatter,
    p_right,
    nrow = 1,
    align = 'h',
    axis = 'tb',
    rel_widths = c(1, strip_ratio)
  )

  cowplot::plot_grid(
    top_row,
    bottom_row,
    ncol = 1,
    align = 'v',
    axis = 'lr',
    rel_heights = c(strip_ratio, 1)
  )
}


## Panels A and B
Scatter plots comparing raw vs preprocessed QC metrics by scanner vendor, each with top/right marginal densities.

In [7]:
# --- Panels A & B: scatter with marginals + export ---
p_a <- build_scatter_with_marginals(
  data = df,
  xvar = 'raw_neighbor_corr',
  yvar = 't1post_neighbor_corr',
  xlab = 'Raw NDC',
  ylab = 'Preprocessed NDC',
  fixed_upper = 1
)

p_b <- build_scatter_with_marginals(
  data = df,
  xvar = 'raw_dwi_contrast',
  yvar = 't1post_dwi_contrast',
  xlab = 'Raw dMRI Contrast',
  ylab = 'Preprocessed dMRI Contrast'
)


panel_dims <- panel_cell_dims
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_a.pdf"), p_a, width_mm = panel_dims$width_mm, height_mm = panel_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_a.jpg"), p_a, width = panel_dims$width_mm, height = panel_dims$height_mm, units = 'mm', dpi = 600, device = 'jpeg')
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_b.pdf"), p_b, width_mm = panel_dims$width_mm, height_mm = panel_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_b.jpg"), p_b, width = panel_dims$width_mm, height = panel_dims$height_mm, units = 'mm', dpi = 600, device = 'jpeg')


Warning message:
"Removed 1088 rows containing missing values or values outside the scale range
(`geom_density()`)."
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_

## Panel C
Vendor-stratified boxplots of median CNR across diffusion shells (b=500, 1000, 2000, 3000).

In [8]:
cnr_long <- df %>%
  select(scanner_manufacturer, CNR1_median, CNR2_median, CNR3_median, CNR4_median) %>%
  filter(scanner_manufacturer %in% vendor_order) %>%
  pivot_longer(cols = starts_with('CNR'), names_to = 'shell', values_to = 'value') %>%
  filter(!is.na(value)) %>%
  mutate(
    scanner_manufacturer = factor(scanner_manufacturer, levels = vendor_order),
    shell = factor(shell, levels = c('CNR1_median', 'CNR2_median', 'CNR3_median', 'CNR4_median'))
  )

cnr_stats <- cnr_long %>%
  group_by(shell, scanner_manufacturer) %>%
  summarise(
    ymin = quantile(value, 0.10, na.rm = TRUE),
    lower = quantile(value, 0.25, na.rm = TRUE),
    middle = quantile(value, 0.50, na.rm = TRUE),
    upper = quantile(value, 0.75, na.rm = TRUE),
    ymax = quantile(value, 0.90, na.rm = TRUE),
    .groups = 'drop'
  )

shell_labels <- c(
  CNR1_median = 'italic(b) == 500',
  CNR2_median = 'italic(b) == 1000',
  CNR3_median = 'italic(b) == 2000',
  CNR4_median = 'italic(b) == 3000'
)

p_c_main <- ggplot(cnr_stats, aes(x = shell, fill = scanner_manufacturer)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = 'identity',
    position = position_dodge2(width = 0.80, preserve = 'single'),
    width = 0.65,
    color = 'black',
    linewidth = 0.20,
    whisker.linewidth = 0.20,
    median.linewidth = 0.28,
    outlier.shape = NA
  ) +
  scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
  scale_x_discrete(labels = function(x) parse(text = unname(shell_labels[x])), drop = FALSE) +
  labs(x = 'dMRI Shell', y = 'Median CNR') +
  theme_pub +
  theme(
    legend.position = 'none',
    aspect.ratio = 1,
    axis.text.x = element_text(angle = 32, hjust = 1, vjust = 1)
  )

p_c <- p_c_main


panel_dims <- panel_cell_dims
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_c.pdf"), p_c, width_mm = panel_dims$width_mm, height_mm = panel_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_c.jpg"), p_c, width = panel_dims$width_mm, height = panel_dims$height_mm, units = 'mm', dpi = 600, device = 'jpeg')


## Panel D
Vendor-stratified boxplots of median b=0 shell tSNR.

In [9]:
tsnr_stats <- df %>%
  select(scanner_manufacturer, CNR0_median) %>%
  filter(scanner_manufacturer %in% vendor_order, !is.na(CNR0_median)) %>%
  mutate(scanner_manufacturer = factor(scanner_manufacturer, levels = vendor_order)) %>%
  group_by(scanner_manufacturer) %>%
  summarise(
    ymin = quantile(CNR0_median, 0.10, na.rm = TRUE),
    lower = quantile(CNR0_median, 0.25, na.rm = TRUE),
    middle = quantile(CNR0_median, 0.50, na.rm = TRUE),
    upper = quantile(CNR0_median, 0.75, na.rm = TRUE),
    ymax = quantile(CNR0_median, 0.90, na.rm = TRUE),
    .groups = 'drop'
  )

p_d_main <- ggplot(tsnr_stats, aes(x = scanner_manufacturer, fill = scanner_manufacturer)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = 'identity',
    width = 0.62,
    color = 'black',
    linewidth = 0.20,
    whisker.linewidth = 0.20,
    median.linewidth = 0.28,
    outlier.shape = NA
  ) +
  scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, guide = 'none') +
  scale_x_discrete(limits = vendor_order, drop = FALSE, expand = expansion(mult = c(0.34, 0.08))) +
  scale_y_continuous(limits = c(0, 45), expand = expansion(mult = c(0, 0))) +
  labs(
    x = 'Scanner Vendor',
    y = 'Median b = 0 tSNR'
  ) +
  theme_pub +
  theme(
    legend.position = 'none',
    axis.text.x = element_text(angle = 35, hjust = 1, vjust = 1),
    aspect.ratio = 2
  )

p_d <- p_d_main


panel_d_dims <- panel_d_dims_final
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_d.pdf"), p_d, width_mm = panel_d_dims$width_mm, height_mm = panel_d_dims$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_d.jpg"), p_d, width = panel_d_dims$width_mm, height = panel_d_dims$height_mm, units = "mm", dpi = 600, device = "jpeg")


## Panel E: Age by software version (by vendor)

Box plots of age by software version for each vendor. Batch values like `anon0982.GE_DV25` are parsed so vendor = GE and software version = DV25. Boxes are colored by vendor. Software versions within each vendor are ordered by mean age.

## Panel F: dMRI IQMs by software version (per vendor)

Vertical box plots for each vendor (GE, Philips, Siemens): two groups of boxes (one per software version). Left facet = Neighboring dMRI Corr. (t1post_neighbor_corr), right facet = dMRI Contrast (t1post_dwi_contrast). Each vendor uses its color (GE orange, Philips green, Siemens blue). X-axis software labels are diagonal. Exports: `Figure2_panel_f.pdf`/`.jpg` (GE), `Figure2_panel_f_Philips.pdf`/`.jpg`, `Figure2_panel_f_Siemens.pdf`/`.jpg`.


In [10]:
# --- Parse batch_device_software into vendor and software_version ---
# Format: anon0982.GE_DV25 -> vendor = GE, software_version = DV25
df_batch <- df %>%
  filter(!is.na(batch_device_software), !is.na(age)) %>%
  mutate(
    batch_suffix = sub("^[^.]+\\.", "", as.character(batch_device_software)),
    vendor_from_batch = sub("_.*", "", batch_suffix),
    software_version = sub("^[^_]+_", "", batch_suffix)
  ) %>%
  filter(vendor_from_batch %in% vendor_order)

# --- Panel E: Age vs software version by vendor (box plots) ---
# Order: vendors grouped (GE, then Philips, then Siemens), within vendor by mean age
order_e <- df_batch %>%
  group_by(vendor_from_batch, software_version) %>%
  summarise(mean_age = mean(age, na.rm = TRUE), .groups = "drop") %>%
  mutate(vendor_ord = match(vendor_from_batch, vendor_order)) %>%
  arrange(vendor_ord, mean_age) %>%
  mutate(software_facet_label = paste(vendor_from_batch, software_version, sep = "::")) %>%
  pull(software_facet_label)

df_e <- df_batch %>%
  mutate(
    vendor_from_batch = factor(vendor_from_batch, levels = vendor_order),
    software_facet = factor(
      paste(vendor_from_batch, software_version, sep = "::"),
      levels = order_e
    )
  )

p_e_main <- ggplot(df_e, aes(x = age, y = software_facet, fill = vendor_from_batch)) +
  geom_boxplot(outlier.size = 0.5, linewidth = 0.25) +
  scale_fill_manual(values = vendor_colors, limits = vendor_order, drop = FALSE, name = "Vendor") +
  scale_y_discrete(labels = function(x) sub("^[^:]*::", "", x), drop = FALSE) +
  labs(x = "Age (years)", y = "Software Version") +
  theme_pub +
  theme(
    legend.position = "none",
    axis.text.y = element_text(angle = 45, hjust = 1, vjust = 1)
  )

p_e <- p_e_main

panel_dims_e <- panel_cell_dims
save_pdf_arial(fs::path(figure2_dir, "panels", "Figure2_panel_e.pdf"), p_e, width_mm = panel_dims_e$width_mm, height_mm = panel_dims_e$height_mm)
ggsave(fs::path(figure2_dir, "panels", "Figure2_panel_e.jpg"), p_e, width = panel_dims_e$width_mm, height = panel_dims_e$height_mm, units = "mm", dpi = 600, device = "jpeg")

# --- Panel F: Neighboring dMRI Corr. and dMRI Contrast by software version, per vendor (GE, Philips, Siemens) ---
panel_dims_f <- list(width_mm = panel_cell_dims$width_mm, height_mm = panel_dims_e$height_mm * 1.12)

for (vendor in vendor_order) {
  sw_order <- df_batch %>%
    filter(vendor_from_batch == vendor) %>%
    group_by(software_version) %>%
    summarise(mean_age = mean(age, na.rm = TRUE), .groups = "drop") %>%
    arrange(mean_age) %>%
    pull(software_version)

  df_f_wide <- df_batch %>%
    filter(vendor_from_batch == vendor) %>%
    mutate(software_version = factor(software_version, levels = sw_order))

  df_f <- df_f_wide %>%
    select(software_version, t1post_neighbor_corr, t1post_dwi_contrast) %>%
    pivot_longer(
      cols = c(t1post_neighbor_corr, t1post_dwi_contrast),
      names_to = "metric",
      values_to = "value"
    ) %>%
    filter(!is.na(value)) %>%
    mutate(
      metric_label = factor(
        if_else(metric == "t1post_neighbor_corr", "Neighboring dMRI Corr.", "dMRI Contrast"),
        levels = c("Neighboring dMRI Corr.", "dMRI Contrast")
      )
    )

  vendor_fill <- unname(vendor_colors[vendor])

  p_f_main <- ggplot(df_f, aes(x = software_version, y = value)) +
    geom_boxplot(fill = vendor_fill, outlier.size = 0.5, linewidth = 0.25) +
    facet_wrap(~ metric_label, ncol = 2, scales = "free_y", strip.position = "top") +
    labs(x = paste0("Software version (", vendor, ")"), y = "Value (Preprocessed)") +
    theme_pub +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      strip.background = element_rect(fill = "grey92", colour = NA),
      strip.text = element_text(size = rel(0.95))
    )

  p_f_v <- p_f_main

  suffix <- if (vendor == "GE") "" else paste0("_", vendor)
  save_pdf_arial(fs::path(figure2_dir, "panels", paste0("Figure2_panel_f", suffix, ".pdf")), p_f_v, width_mm = panel_dims_f$width_mm, height_mm = panel_dims_f$height_mm)
  ggsave(fs::path(figure2_dir, "panels", paste0("Figure2_panel_f", suffix, ".jpg")), p_f_v, width = panel_dims_f$width_mm, height = panel_dims_f$height_mm, units = "mm", dpi = 600, device = "jpeg")
  if (vendor == "GE") p_f <- p_f_v
}
